In [1]:
import pandas as pd
from sklearn.metrics import auc, roc_curve
import yaml
import os
from sklearn.metrics import auc
from statistics import mean
import re

In [2]:
def calculate_auc_avg(tickers, predictions, true_values):
    predictions = predictions.sort_values(by='fecha')
    true_values = true_values.sort_values(by='fecha')
    
    
    # renombro las columnas
    column_names = {}
    for column in predictions.columns:
        if column != 'fecha':
            column_names[column] = f'{column}_proba'
    
    predictions = predictions.rename(columns=column_names)
    
    column_names = {}
    for column in true_values.columns:
        if column != 'fecha':
            column_names[column] = f'{column}_true'
    
    true_values = true_values.rename(columns=column_names)
    performance = pd.concat(
        [
            predictions,
            true_values
        ], axis=1, join="inner"
    )
    
    auc_list = []
    for ticker in tickers:
        y_true = performance[performance[f'{ticker}_true'].notna()][f'{ticker}_true']
        y_pred = performance[performance[f'{ticker}_proba'].notna()][f'{ticker}_proba']
        
        fpr, tpr, thresholds = roc_curve(y_true, y_pred)
        auc_score = auc(fpr, tpr)
        
        auc_list.append(auc_score)

    return mean(auc_list)

def max_drawdown(serie):
    max_valor_acumulado = serie[0]
    max_dd = 0

    for valor_actual in serie[1:]:
        if valor_actual > max_valor_acumulado:
            max_valor_acumulado = valor_actual
        else:
            dd = (max_valor_acumulado - valor_actual) / max_valor_acumulado
            if dd > max_dd:
                max_dd = dd

    return max_dd

In [3]:
with open('configs/project_config.yml', 'r') as archivo:
    config = yaml.safe_load(archivo)

tickers = config["tickers"] 
tickers

['YPF', 'BBAR', 'BMA', 'CEPU', 'EDN', 'TEO', 'LOMA']

In [4]:
stat_tickers_dict = {}

for ticker in tickers:
    df_ticker  = pd.read_csv(f'./data/{ticker}.csv')
    stat_tickers_dict[ticker] = df_ticker.Close.mean()
stat_tickers_dict

{'YPF': 7.220402563588944,
 'BBAR': 2.9153070466189024,
 'BMA': 14.565118127423258,
 'CEPU': 3.4468957281723442,
 'EDN': 6.604849040780936,
 'TEO': 5.889229579464842,
 'LOMA': 5.300568032679161}

In [5]:
results_dict = {}

for path in os.listdir('./data'):
    if not path.endswith('.csv') and path.startswith('mode_train'):
        print(path)
        results_dict[path] = {}

        try:
            initial_wallet_value = 100
            wallet = pd.read_csv(os.path.join('./data', path, 'wallet.csv'))
            final_wallet_value = wallet.tail(1).iloc[0]['wallet']

            results_dict[path]['wallet'] = ((final_wallet_value - initial_wallet_value) / initial_wallet_value) * 100
            results_dict[path]['max_drawdown'] = max_drawdown(wallet['wallet'])
        except:
            results_dict[path]['wallet'] = 0

        try:
            orders = pd.read_csv(os.path.join('./data', path, 'orders.csv'))
            results_dict[path]['buys'] = orders[orders['open_date'].notna()].shape[0]
            results_dict[path]['sells'] = orders[orders['close_date'].notna()].shape[0]

            profit_tickers = orders.groupby('ticker')['profit'].sum().reset_index()
            profit_tickers['mean'] = profit_tickers.ticker.map(stat_tickers_dict)
            profit_tickers['profit_norm'] = profit_tickers.profit / profit_tickers['mean']
            results_dict[path]['avg_incomes_norm'] = profit_tickers.profit_norm.sum()
            
           
            avg_incomes = orders.groupby('ticker')['profit'].sum().mean()
            results_dict[path]['avg_incomes'] = avg_incomes
            

            results_dict[path]['good_operations'] = orders[orders['profit'] > 0].shape[0]
            results_dict[path]['bad_operations'] = orders[orders['profit'] <= 0].shape[0]

            results_dict[path]['total_operations'] = orders.shape[0]

            results_dict[path]['winning_rate'] = results_dict[path]['good_operations'] / (results_dict[path]['good_operations'] + results_dict[path]['bad_operations']) 
            
        except:
            results_dict[path]['buys'] = 0
            results_dict[path]['sells'] = 0
            avg_incomes = 0
            results_dict[path]['avg_incomes'] = 0
            results_dict[path]['good_operations'] = 0
            results_dict[path]['bad_operations'] = 0
            results_dict[path]['winning_rate'] = 0

        try:
            train_results = pd.read_csv(os.path.join('./data', path, 'train_results.csv'))
            avg_train_auc = train_results['auc'].mean()
            results_dict[path]['avg_train_auc'] = avg_train_auc
        except:
            results_dict[path]['avg_train_auc'] = 0
            
        try:
            stock_predictions = pd.read_csv(os.path.join('./data', path, 'stock_predictions.csv'))
            true_values = pd.read_csv(os.path.join('./data', path, 'stock_true_values.csv'))

            avg_auc_score = calculate_auc_avg(tickers, stock_predictions, true_values)
            results_dict[path]['avg_test_auc'] = avg_auc_score
        except:
            results_dict[path]['avg_test_auc'] = 0

results = pd.DataFrame.from_dict(results_dict, orient='index')

mode_train-model_gradient_boosting-trainwindow_114-trainperiod_14-tradingstrategy_strategies.bband_strategy
mode_train-model_gradient_boosting-trainwindow_114-trainperiod_14-tradingstrategy_strategies.macd_strategy
mode_train-model_gradient_boosting-trainwindow_114-trainperiod_14-tradingstrategy_strategies.ma_strategy
mode_train-model_gradient_boosting-trainwindow_114-trainperiod_14-tradingstrategy_strategies.ml_strategy
mode_train-model_gradient_boosting-trainwindow_114-trainperiod_30-tradingstrategy_strategies.bband_strategy
mode_train-model_gradient_boosting-trainwindow_114-trainperiod_30-tradingstrategy_strategies.macd_strategy
mode_train-model_gradient_boosting-trainwindow_114-trainperiod_30-tradingstrategy_strategies.ma_strategy
mode_train-model_gradient_boosting-trainwindow_114-trainperiod_30-tradingstrategy_strategies.ml_strategy
mode_train-model_gradient_boosting-trainwindow_114-trainperiod_7-tradingstrategy_strategies.bband_strategy
mode_train-model_gradient_boosting-trainwin

Metrica = Avg Incomes norm / (1+Max Drawdown) × log(1+Numero de Operaciones)

In [6]:
import numpy as np
results['performance'] = ((results['avg_incomes_norm'])/(1 + results['max_drawdown'])) * np.log(1 + results['total_operations'])

results = results.sort_values(by=['performance'], ascending=[False])[
    [
        'performance',
        'winning_rate', 
        'avg_incomes_norm', 
        'max_drawdown',
        'good_operations',
        'bad_operations',
        'avg_train_auc',
        'avg_test_auc',
        'wallet',
    ]
]

Calculo el avg del profit de cada activo y a eso lo divido por el std de ese activo

luego de esto me quedarian 7 valores (uno por cada activo). Sumo todo eso y lo meto en la formula

In [7]:
results = results.reset_index().rename(columns={'index':'config'})

# With / without model

In [8]:
results = results[(results.performance > 0)]
print(results.shape)

(194, 10)


las metricas seran calculadas solo con aquellas estrategias cuya performance sea positiva.

In [9]:
results['window'] = results['config'].apply(lambda x: re.search(r"trainwindow_(.*?)-", x).group(1)).astype(int)


results['with_model'] = np.where(results['window']!=0, 1, 0)

perfo = pd.DataFrame(
    results.groupby('with_model').agg({'performance':['median', 'mean', 'std', 'min', 'max']})
)
perfo

performance                                         
                median      mean       std       min        max
with_model                                                     
0             7.505103  7.882237  1.214957  6.900570   9.241038
1             5.242953  5.215451  2.873075  0.032502  13.425208

- Media y Mediana: La media y mediana de las estrategias sin modelo (7.88 y 7.51 respectivamente) son mayores en comparación con las estrategias con modelo, que (5.21 y 5.24). Esto indica que, en promedio, las estrategias sin modelo parecen generar un mejor rendimiento que las estrategias que usan un modelo.

- Desviación estándar: La desviación estándar es mucho más alta en las estrategias con modelo (2.87) en comparación con las sin modelo (1.21). Esto sugiere que las estrategias con modelo presentan una mayor volatilidad en su rendimiento, con resultados más dispersos, mientras que las estrategias sin modelo tienen un rendimiento más consistente. 

- Rango (Mínimo y Máximo): La estrategia con modelo tiene un rendimiento mínimo muy bajo (0.03), lo que sugiere que en algunas ocasiones puede haber casi nulo o muy bajo rendimiento, mientras que el rendimiento máximo es alto (13.42), lo que refleja un potencial de altos beneficios.
Por otro lado, las estrategias sin modelo tienen un mínimo de 6.90, lo cual es significativamente más alto y consistente, con un máximo de 9.24, que también es alto pero menor al de las estrategias con modelo.

- Conclusión: Las estrategias sin modelo parecen ofrecer un rendimiento más consistente y con menor volatilidad, con un mínimo rendimiento bastante sólido.
Por otra parte, las estrategias con modelo muestran una mayor volatilidad en los resultados, con un rango muy amplio desde rendimientos bajos hasta algunos muy altos. Podría haber un mayor riesgo asociado, pero también existe un mayor potencial de recompensa.

In [10]:

pd.concat([
    results[
        (results.with_model==0) & (results.config.str.contains('macd_strategy'))].set_index('config').sort_values(by='performance', ascending=False).head(1),
    results[(results.with_model==1) & (results.config.str.contains('macd_strategy'))].set_index('config').sort_values(by='performance', ascending=False).head(1),

    results[(results.with_model==0) & (results.config.str.contains('ma_strategy'))].set_index('config').sort_values(by='performance', ascending=False).head(1),
    results[(results.with_model==1) & (results.config.str.contains('ma_strategy'))].set_index('config').sort_values(by='performance', ascending=False).head(1),

    results[(results.with_model==0) & (results.config.str.contains('bband_strategy'))].set_index('config').sort_values(by='performance', ascending=False).head(1),
    results[(results.with_model==1) & (results.config.str.contains('bband_strategy'))].set_index('config').sort_values(by='performance', ascending=False).head(1),
    
    results[(results.with_model==1) & (results.config.str.contains('ml_strategy'))].set_index('config').sort_values(by='performance', ascending=False).head(1),
])[['performance', 'winning_rate', 'avg_incomes_norm', 'max_drawdown', 'good_operations', 'bad_operations']]


,performance,winning_rate,avg_incomes_norm,max_drawdown,good_operations,bad_operations
config,,,,,,
mode_train-model_None-trainwindow_0-trainperiod_0-tradingstrategy_strategies.macd_strategy,9.241038,0.520548,2.254110,0.053138,38,35
mode_train-model_logistic_regression-trainwindow_689-trainperiod_30-tradingstrategy_strategies.macd_strategy,10.101355,0.564516,2.539780,0.045666,35,27
mode_train-model_None-trainwindow_0-trainperiod_0-tradingstrategy_strategies.ma_strategy,7.505103,0.500000,1.775712,0.134687,57,57
mode_train-model_logistic_regression-trainwindow_689-trainperiod_7-tradingstrategy_strategies.ma_strategy,12.712534,0.500000,2.908032,0.068924,51,51
mode_train-model_None-trainwindow_0-trainperiod_0-tradingstrategy_strategies.bband_strategy,6.900570,0.549451,1.667422,0.107890,50,41
mode_train-model_logistic_regression-trainwindow_689-trainperiod_7-tradingstrategy_strategies.bband_strategy,10.092617,0.551282,2.404369,0.055556,43,35
mode_train-model_logistic_regression-trainwindow_689-trainperiod_7-tradingstrategy_strategies.ml_strategy,13.425208,0.504762,3.052479,0.068744,53,52


1. Performance (Rendimiento):
En todos los casos, el rendimiento es mayor cuando se utiliza el modelo de machine learning (Logistic Regression) en comparación con la versión sin modelo:
MACD strategy: Con modelo, el rendimiento es 10.10 vs 9.24 sin modelo.
MA strategy: Con modelo, el rendimiento es 12.71 vs 7.50 sin modelo.
BBand strategy: Con modelo, el rendimiento es 10.09 vs 6.90 sin modelo.
Conclusión: Implementar el modelo de machine learning mejora el rendimiento general en todas las estrategias.

2. Winning Rate (Tasa de éxito):
La tasa de éxito mejora ligeramente en la estrategia MACD (0.56 con modelo vs 0.52 sin modelo), y permanece estable en las demás:
MA strategy: 0.50 en ambos casos.
BBand strategy: Apenas varía (0.55 con modelo vs 0.55 sin modelo).
Conclusión: El modelo no parece tener un impacto significativo en la tasa de éxito en las estrategias de MA y BBand, pero sí mejora levemente en la estrategia MACD.

3. Avg Incomes Normalized (Ingresos Promedios Normalizados):
En todos los casos, los ingresos promedios normalizados son mayores con el modelo de machine learning:
MACD strategy: 2.54 con modelo vs 2.25 sin modelo.
MA strategy: 2.90 con modelo vs 1.77 sin modelo.
BBand strategy: 2.40 con modelo vs 1.66 sin modelo.
Conclusión: El modelo de machine learning mejora significativamente los ingresos normalizados en todas las estrategias.

4. Max Drawdown (Pérdida Máxima):
En general, el drawdown es menor en las estrategias con el modelo, lo que implica menos riesgo en términos de caídas de capital:
MACD strategy: 0.045 con modelo vs 0.053 sin modelo.
MA strategy: 0.068 con modelo vs 0.134 sin modelo.
BBand strategy: 0.055 con modelo vs 0.107 sin modelo.
Conclusión: Las estrategias con modelo parecen tener un riesgo más controlado y presentan menores caídas en su cartera.

5. Good Operations (Operaciones Ganadoras) vs. Bad Operations (Operaciones Perdedoras):
Las estrategias con el modelo suelen tener menos operaciones en general (tanto ganadoras como perdedoras), pero con mejores resultados en términos de rendimiento global.
MACD strategy: Con modelo, 35 operaciones ganadoras vs 38 sin modelo, pero con menos operaciones perdedoras (27 con modelo vs 35 sin modelo).
MA strategy: Con modelo, 51 ganadoras y 51 perdedoras vs 57 ganadoras y 57 perdedoras sin modelo.
BBand strategy: Con modelo, 43 ganadoras vs 50 sin modelo, pero menos operaciones perdedoras (35 con modelo vs 41 sin modelo).
Conclusión: Las estrategias con modelo tienden a reducir la cantidad de operaciones, lo que podría indicar una mayor selectividad en la apertura de posiciones, pero estas operaciones son generalmente más rentables.


Implementar un modelo de machine learning mejora significativamente el rendimiento de las estrategias de trading, especialmente en términos de performance e ingresos normalizados, y también reduce el drawdown máximo, lo que implica un menor riesgo en las caídas de capital.
A pesar de que el número de operaciones ganadoras y perdedoras tiende a ser menor con el modelo, las operaciones que realiza son generalmente más rentables y consistentes.
La tasa de éxito (winning rate) se mantiene similar o mejora ligeramente, lo que sugiere que el modelo no compromete la efectividad general.
El modelo de machine learning parece seleccionar mejor las oportunidades de trading, mostrando una mayor selectividad y contribuyendo a un rendimiento más eficiente con menor riesgo.

# Window

In [11]:
results = results[(results.with_model == 1)]

In [ ]:
pd.DataFrame(
    results.groupby('window').agg({'performance':['median', 'mean', 'std', 'min', 'max']})
).sort_values(by=('window'), ascending=False)

Mediana:
La mediana más alta se obtiene con una ventana de 461 días (7.062494), lo que indica que este tamaño de ventana proporciona el rendimiento central más sólido. Las ventanas de 689 días (6.637734) y 922 días (6.423080) siguen de cerca, lo que sugiere que, aunque las ventanas más grandes todavía ofrecen un rendimiento sólido, hay una ligera disminución conforme el tamaño de la ventana aumenta. Las ventanas más pequeñas, como las de 228 días (5.530853), muestran una mediana considerablemente más baja, y las ventanas de 114 (1.247758) y 76 días (3.053821) tienen las medianas más bajas, lo que indica un rendimiento central deficiente.

Media:
La media más alta también se observa con una ventana de 461 días (6.661510), seguida por la ventana de 689 días (6.452212) y 922 días (6.241324). Estos resultados indican que las ventanas medianas y grandes tienen un rendimiento promedio relativamente alto y consistente. Por otro lado, las ventanas más pequeñas (114 y 76 días) muestran medias muy bajas (1.495259 y 2.934197, respectivamente), lo que sugiere que estas ventanas no permiten al modelo capturar patrones significativos de manera eficiente.

Desvío estándar:
El desvío estándar más bajo se observa con la ventana de 461 días (1.937751), lo que indica que esta ventana genera resultados más consistentes y menos dispersos. Las ventanas de 922 días (2.249263) y 228 días (2.425922) tienen una desviación estándar moderada, lo que implica una mayor variabilidad, pero aún dentro de un rango aceptable. Por otro lado, la ventana de 689 días (3.029967) presenta la mayor desviación estándar, lo que refleja una mayor dispersión de los resultados. Las ventanas más pequeñas (76 días y 114 días) tienen desviaciones estándar relativamente bajas, pero esto se debe a que sus resultados son consistentemente bajos.

Mínimo y Máximo:
En cuanto al rendimiento mínimo, la ventana de 114 días tiene el peor valor (0.032502), lo que indica que puede haber casos de rendimiento extremadamente bajo cuando se utiliza una ventana pequeña. De manera similar, la ventana de 76 días también tiene un rendimiento mínimo muy bajo (0.124406). En cambio, las ventanas más grandes (922 días, 689 días y 461 días) muestran mínimos significativamente más altos (2.261863, 1.634875 y 1.766991, respectivamente), lo que indica que estas ventanas permiten evitar caídas drásticas en el rendimiento.
El rendimiento máximo más alto se observa con la ventana de 689 días (13.425208), lo que sugiere que esta ventana ofrece el mayor potencial de ganancias en los mejores casos. Sin embargo, la ventana de 461 días (9.438873) ofrece un máximo considerable, mientras que la ventana de 922 días (10.802135) también proporciona un rendimiento máximo respetable. Las ventanas más pequeñas muestran máximos más bajos, lo que indica un menor potencial de retorno.

Conclusión:
El tamaño de ventana de 461 días parece ofrecer el mejor equilibrio entre rendimiento central (mediana), rendimiento promedio (media), y consistencia (desviación estándar baja). Aunque ventanas más grandes como la de 689 días ofrecen un mayor potencial de rendimiento máximo, también vienen acompañadas de una mayor variabilidad en los resultados. Las ventanas más pequeñas (114 y 76 días) no capturan suficientes datos para proporcionar un rendimiento sólido, mostrando tanto bajos rendimientos centrales como un menor potencial de ganancias.

Por lo tanto, si se busca maximizar el rendimiento y estabilidad, la ventana de 461 días parece ser la opción más adecuada, mientras que ventanas más grandes como la de 689 días pueden ser apropiadas si se desea asumir mayor riesgo a cambio de un posible rendimiento más alto. Las ventanas pequeñas son ineficaces en este contexto.

# trainperiod

In [ ]:
results['trainperiod'] = results['config'].apply(lambda x: re.search(r'trainperiod_(.*?)-', x).group(1))
pd.DataFrame(
    results.groupby('trainperiod').agg({'performance':['median', 'mean', 'std', 'min', 'max']})
).sort_values(by=('performance','median'), ascending=False)

Mediana:
La mediana más alta se observa con un periodo de entrenamiento de 7 días (6.040182), lo que indica que entrenar con mayor frecuencia tiende a ofrecer un mejor rendimiento central. Los periodos de entrenamiento de 30 días (5.312385) y 14 días (4.880899) muestran medianas más bajas, lo que sugiere que, a medida que se alarga el periodo de entrenamiento, el rendimiento central disminuye ligeramente.

Media:
Las medias siguen una tendencia similar, con el periodo de 7 días presentando la media más alta (5.790942), seguido de 30 días (5.341965) y 14 días (4.467036). Esto sugiere que, en promedio, entrenar con mayor frecuencia proporciona mejores resultados. La diferencia entre los periodos de 7 y 30 días es pequeña, pero el periodo de 14 días muestra un rendimiento promedio más bajo.

Desvío estándar:
El desvío estándar más alto se observa en el periodo de 7 días (3.276232), lo que indica que aunque este periodo ofrece el mejor rendimiento, también tiene la mayor volatilidad en los resultados. El periodo de 30 días tiene un desvío estándar moderado (2.824728), y el periodo de 14 días muestra la menor dispersión (2.273541), lo que indica que es más consistente, aunque con menor rendimiento promedio.

Mínimo y Máximo:
El máximo rendimiento más alto se obtiene con el periodo de 7 días (13.425208), lo que sugiere que este periodo ofrece el mayor potencial de rendimiento en los mejores casos. Los periodos de 30 días (10.417989) y 14 días (8.618804) muestran máximos más bajos, lo que indica que entrenar con menos frecuencia reduce el potencial de máximos altos. En cuanto al mínimo rendimiento, el periodo de 30 días tiene el peor valor (0.032502), seguido por 7 días (0.059538) y 14 días (0.094091), lo que muestra que hay un riesgo moderado de caídas bajas en todos los periodos, pero ninguno es extremo.

Conclusión:
Entrenar con mayor frecuencia (cada 7 días) ofrece el mejor rendimiento tanto en términos de mediana como de media, y también tiene el mayor potencial de rendimiento máximo. Sin embargo, viene con el costo de una mayor volatilidad en los resultados. Entrenar cada 30 días es un equilibrio razonable, con un buen rendimiento promedio, pero menor volatilidad en comparación con 7 días. El periodo de 14 días, aunque el más consistente en términos de desvío estándar, tiene los rendimientos promedio y máximos más bajos, lo que lo convierte en la opción más segura, pero menos rentable.


# Model

In [ ]:
results['model'] = results['config'].apply(lambda x: re.search(r'model_(.*?)-', x).group(1))
pd.DataFrame(
    results.groupby('model').agg({'performance':['median', 'mean', 'std', 'min', 'max']})
).sort_values(by=('performance','median'), ascending=False)

Mediana:
La mediana más alta se observa con el modelo de logistic regression (6.445060), lo que indica que este modelo ofrece el mejor rendimiento central. El modelo de random forest tiene una mediana de 5.068768, mientras que el modelo de gradient boosting presenta la mediana más baja (4.531844), lo que sugiere que este último modelo ofrece un rendimiento central menos robusto en comparación con los otros dos.

Media:
La media más alta también corresponde al modelo de logistic regression (6.072898), seguido por random forest (4.978542) y gradient boosting (4.694463). Esto sugiere que, en promedio, logistic regression proporciona el mejor rendimiento, mientras que los otros dos modelos tienen un rendimiento más modesto. El modelo de random forest y gradient boosting están bastante cercanos en términos de rendimiento promedio, pero ambos se sitúan por debajo de logistic regression.

Desvío estándar:
El desvío estándar más alto se encuentra en el modelo de logistic regression (3.073485), lo que indica una mayor volatilidad en los resultados, aunque ofrece el mejor rendimiento central. El modelo de random forest tiene un desvío estándar de 2.815325, lo que sugiere una variabilidad moderada en sus resultados. Por otro lado, el modelo de gradient boosting muestra el menor desvío estándar (2.608003), lo que indica que sus resultados son los más consistentes, aunque con menor rendimiento.

Mínimo y Máximo:
En cuanto al rendimiento máximo, el modelo de logistic regression tiene el valor más alto (13.425208), lo que sugiere un gran potencial de ganancias en los mejores casos. El modelo de gradient boosting presenta un máximo de 11.755405, mientras que el modelo de random forest tiene el máximo más bajo (10.802135).
En cuanto al rendimiento mínimo, el modelo de gradient boosting tiene el peor valor (0.032502), lo que indica que este modelo puede llegar a generar rendimientos muy bajos en ciertos casos. El modelo de random forest tiene un mínimo de 0.059538, mientras que el logistic regression presenta el mejor mínimo (0.284693), lo que sugiere que, en los peores casos, tiende a evitar rendimientos extremadamente bajos.

Conclusión:
El modelo de logistic regression es el que ofrece el mejor rendimiento central (mediana) y promedio (media), y también tiene el mayor potencial de rendimiento máximo. Sin embargo, tiene una mayor volatilidad en los resultados (desvío estándar más alto), lo que sugiere que este modelo podría generar tanto buenos rendimientos como resultados más dispersos.

El modelo de random forest presenta un rendimiento moderado tanto en mediana como en media, con una volatilidad intermedia. Es una opción razonable para quienes buscan un equilibrio entre rendimiento y consistencia.

El modelo de gradient boosting tiene el rendimiento más bajo, pero ofrece la mayor consistencia (menor desvío estándar), lo que lo convierte en la opción más predecible, aunque con menor potencial de rendimiento.

Para maximizar tanto el rendimiento como el potencial máximo, el modelo de logistic regression parece ser la mejor opción, aunque con mayor riesgo. Si se busca más estabilidad y menor variabilidad, el modelo de gradient boosting podría ser más adecuado.

# Strategy

In [ ]:
results['tradingstrategy'] = results['config'].apply(lambda x: re.search(r'tradingstrategy_(.*)', x).group(1))
pd.DataFrame(
    results.groupby('tradingstrategy').agg({'performance':['median', 'mean', 'std', 'min', 'max']})
).sort_values(by=('performance','median'), ascending=False)

Mediana:
La mediana más alta corresponde a la estrategia ml_strategy (6.575939), lo que indica que esta estrategia ofrece el rendimiento central más robusto. Le sigue la estrategia ma_strategy con una mediana de 5.762674, lo que sugiere un rendimiento central sólido pero ligeramente inferior. La estrategia macd_strategy tiene una mediana de 4.667082, y la bband_strategy presenta la mediana más baja (3.721428), indicando que esta última estrategia tiene el rendimiento central más débil.

Media:
En cuanto a la media, la estrategia ml_strategy lidera nuevamente con un valor de 6.231975, confirmando que en promedio es la que genera mejores resultados. La estrategia ma_strategy muestra una media de 5.852659, lo que la coloca cerca en términos de rendimiento promedio. Macd_strategy y bband_strategy presentan medias más bajas, de 4.654121 y 4.316034, respectivamente, lo que refleja que, en promedio, generan resultados inferiores a las dos primeras estrategias.

Desvío estándar:
El desvío estándar más alto se observa en la estrategia macd_strategy (3.068973), lo que indica que sus resultados son los más volátiles y dispersos. La estrategia ml_strategy tiene un desvío estándar ligeramente inferior (3.043121), lo que también sugiere cierta volatilidad, pero con mejores resultados centrales. Ma_strategy muestra una volatilidad moderada (2.735122), mientras que la bband_strategy tiene el desvío estándar más bajo (2.191217), lo que indica que es la estrategia más consistente, aunque con un rendimiento más limitado.

Mínimo y Máximo:
La estrategia ml_strategy presenta tanto el máximo rendimiento más alto (13.425208) como el mínimo rendimiento más bajo (0.085172), lo que sugiere que esta estrategia tiene un gran potencial pero también puede producir algunos resultados muy bajos en ciertos escenarios. La estrategia ma_strategy sigue de cerca, con un máximo de 12.712534 y un mínimo de 0.284693, lo que indica que tiene un potencial significativo y evita rendimientos extremadamente bajos.
La estrategia macd_strategy tiene un máximo más bajo (10.101355) y un mínimo más bajo aún (0.032502), lo que implica que es la estrategia con mayor riesgo de bajo rendimiento. Finalmente, la bband_strategy tiene un máximo de 10.092617 y un mínimo de 0.566930, lo que sugiere un menor potencial de rendimiento máximo, pero también un riesgo más controlado en cuanto a resultados negativos.

Conclusión:
La estrategia ml_strategy ofrece el mejor rendimiento central (mediana) y promedio (media), así como el mayor potencial de rendimiento máximo, aunque con una volatilidad moderada en los resultados. Esto la convierte en la mejor opción si se busca maximizar el rendimiento a pesar de aceptar cierto nivel de riesgo.

La estrategia ma_strategy es una opción fuerte, con un rendimiento promedio cercano a ml_strategy y una volatilidad moderada, lo que la hace atractiva para quienes buscan un buen equilibrio entre potencial de rendimiento y riesgo controlado.

La estrategia macd_strategy tiene el rendimiento más volátil, con un alto riesgo de bajos rendimientos y un menor potencial máximo, lo que la convierte en una opción menos estable. Finalmente, la bband_strategy es la más consistente, con menor volatilidad, pero a costa de un rendimiento central y potencial máximo más limitados.

Para quienes buscan maximizar ganancias y aceptan cierta volatilidad, la ml_strategy es la opción recomendada. Si se prefiere una estrategia más predecible y consistente, la bband_strategy podría ser más adecuada.